# CARMA Classification: Data Prep & Split Construction

Builds the **canonical, user-level, balanced train/test split** used by both classification experiments described in the paper (§7 Experiments):

- classical TF-IDF classifiers (post-level instances, label inherited from author)
- fine-tuned PLMs (user-level, posts concatenated per user)

Both experiments must share the **same** user-level train/test assignment per condition so results are comparable and there is no leakage. This notebook does data prep only — no model training. Review the sample-size tables below before we wire the experiment scripts to consume the resulting split file.

Source: `data/posts/v1/reddit-preprocessed.csv` (post-level, columns: `user_id`, `class`, `text`).

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

RAW        = "../data/posts/v1/reddit-preprocessed.csv"
OUT_DIR    = Path("../data/splits")
SEED       = 42
TEST_SIZE  = 0.10
MIN_USERS  = 30   # minimum positive users required to include a condition

OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 120)

## 1. Load raw post-level data

In [2]:
df = pd.read_csv(RAW, keep_default_na=False, usecols=["user_id", "class", "text"])
print(f"{len(df):,} posts, {df['user_id'].nunique():,} unique users")
df.head()

1,115,379 posts, 9,149 unique users


,user_id,class,text
0,No_Entry_2175,adhd,الله عليك يا عبغفور ماشي بمبدا معنديش حاجة اخس...
1,Mysterious-Pace-5657,bpd,غلط عن غلط يفرق الضرب والتعنيف من المحرمات في ...
2,Mfkenkaneki,depression,اقل مسلم بيكفرك من غير ما يعرف اسمك حتي ، شوية...
3,PEACEandLOVEEEEE,depression,كلمت ابويا عن المثليين قولتله هو انا عندي سؤال...
4,Flimsy-Leopard-8022,control,اجهزه تعدين من فين اقدر اشتريها ؟ لو حد يقدر ي...


In [3]:
# raw per-row class distribution (posts, not users) — includes blank class for
# posts that didn't trigger any pattern match during longitudinal history retrieval
df["class"].value_counts(dropna=False)

class
control             264469
depression          251809
                    186398
adhd                101107
anxiety              90082
sleep_disorder       60666
ocd                  39413
autism               32292
panic                20849
ptsd                 15627
suicidal             12266
bpd                  10735
bipolar               8894
schizophrenia         7703
eating_disorder       5386
paranoid_pd           4011
did                   1914
narcissistic_pd        892
trichotillomania       461
avoidant_pd            405
Name: count, dtype: int64

## 2. Derive a single label per user

`class` is assigned **per post**, not per user. For a diagnosed user, only the post(s) that matched the self-disclosure pattern carry the condition label — the rest of that user's retrieved history (posts that didn't trigger any pattern) come through as `control` or blank in this file. Naively taking `class` at face value would misclassify diagnosed users' own non-matching posts as `control`, which would leak diagnosed users into the control pool.

Check: does any user ever carry **two different real conditions**, or is the only mixing pattern "one condition + control/blank filler rows"?

In [4]:
per_user_classes = df.groupby("user_id")["class"].apply(lambda s: tuple(sorted(set(s))))
multi = per_user_classes[per_user_classes.apply(len) > 1]
print(f"{len(multi)} users have more than one distinct class value across their posts")

real_multi_condition = multi[multi.apply(lambda s: len(set(s) - {"", "control"}) > 1)]
print(f"{len(real_multi_condition)} of those mix two or more *real* conditions (true comorbidity at the post-label level)")

# most common combos, for a quick sanity look
multi.value_counts().head(10)

333 users have more than one distinct class value across their posts
0 of those mix two or more *real* conditions (true comorbidity at the post-label level)


class
(control, depression)        99
(, control)                  98
(anxiety, control)           31
(adhd, control)              27
(control, sleep_disorder)    24
(control, ocd)               15
(control, panic)              8
(autism, control)             8
(bipolar, control)            5
(bpd, control)                4
Name: count, dtype: int64

None of the mixed-label users combine two real conditions — the only mixing is a real condition alongside `control`/blank filler rows from that same user's non-matching posts. So we can safely collapse to one label per user:

- if any post carries a real condition → the user's label is that condition
- else if any post is explicitly `control` → the user's label is `control`
- else (every post is blank) → `unknown`, excluded from all downstream experiments

The `unknown` bucket exists because some scraped users never got an explicit `control` stamp on any post; treating them as control by default would be an unverified assumption, so we drop them instead.

In [5]:
def collapse_to_user_label(classes):
    real_conditions = set(classes) - {"", "control"}
    if real_conditions:
        assert len(real_conditions) == 1, f"unexpected multi-condition user: {classes}"
        return next(iter(real_conditions))
    if "control" in classes:
        return "control"
    return "unknown"

user_label = per_user_classes.apply(collapse_to_user_label)
user_label.name = "label"

print(f"{(user_label == 'unknown').sum()} users excluded as 'unknown' (no explicit control or condition label)")
user_label.value_counts()

662 users excluded as 'unknown' (no explicit control or condition label)


label
control             3845
depression          1831
unknown              662
anxiety              644
adhd                 643
sleep_disorder       357
ocd                  311
autism               227
panic                147
bpd                   98
ptsd                  92
suicidal              80
bipolar               76
schizophrenia         45
eating_disorder       41
paranoid_pd           24
did                   11
trichotillomania       9
narcissistic_pd        3
avoidant_pd            3
Name: count, dtype: int64

## 3. Sample sizes per condition (review before approving)

This is the number of **users currently available** in `reddit-preprocessed.csv` per condition, independent of any balancing. Note this file is the raw pattern-pipeline output (9,149 users) and is larger than the curated 3,080-user corpus reported in the paper — treat these as upper bounds on what we can use, not the published corpus counts.

In [6]:
n_control = (user_label == "control").sum()

sizes = (
    user_label[user_label.isin(["unknown"]) == False]
    .value_counts()
    .rename("n_users")
    .to_frame()
)
sizes = sizes.drop(index="control")
sizes["n_control_available"] = n_control
sizes["n_balanced_per_class"] = sizes[["n_users", "n_control_available"]].min(axis=1)
sizes["meets_min_users"] = sizes["n_users"] >= MIN_USERS
sizes = sizes.sort_values("n_users", ascending=False)

sizes.to_csv(OUT_DIR / "sample_sizes.csv")
sizes

,n_users,n_control_available,n_balanced_per_class,meets_min_users
label,,,,
depression,1831,3845,1831,True
anxiety,644,3845,644,True
adhd,643,3845,643,True
sleep_disorder,357,3845,357,True
ocd,311,3845,311,True
autism,227,3845,227,True
panic,147,3845,147,True
bpd,98,3845,98,True
ptsd,92,3845,92,True


In [7]:
excluded = sizes[~sizes["meets_min_users"]]
print(f"Excluded (< {MIN_USERS} positive users): {', '.join(excluded.index)}")

CONDITIONS = sizes[sizes["meets_min_users"]].index.tolist()
print(f"\n{len(CONDITIONS)} conditions included in experiments:")
print(CONDITIONS)

Excluded (< 30 positive users): paranoid_pd, did, trichotillomania, narcissistic_pd, avoidant_pd

13 conditions included in experiments:
['depression', 'anxiety', 'adhd', 'sleep_disorder', 'ocd', 'autism', 'panic', 'bpd', 'ptsd', 'suicidal', 'bipolar', 'schizophrenia', 'eating_disorder']


## 4. Balanced 1:1 user-level split (per condition)

For each included condition:
1. Sample `n = min(n_positive, n_control)` control users (without replacement), seeded.
2. Split the positive set and the sampled control set **independently** into 90/10 train/test at the user level, so both splits stay balanced.
3. Control users are resampled per condition (a control user may appear in more than one condition's split, but never appears as both `train` and `test` *within the same condition*).

This produces one manifest with `(condition, user_id, y, split)` — no text, no leakage across splits. Downstream scripts join this back to `reddit-preprocessed.csv` by `user_id` to build post-level (classical) or user-aggregated (PLM) inputs.

In [8]:
control_pool = user_label.index[user_label == "control"].to_numpy()

def build_condition_split(condition):
    pos_pool = user_label.index[user_label == condition].to_numpy()
    n = min(len(pos_pool), len(control_pool))

    rng = np.random.default_rng(SEED)
    pos_sample = rng.choice(pos_pool, size=n, replace=False)
    neg_sample = rng.choice(control_pool, size=n, replace=False)

    pos_train, pos_test = train_test_split(pos_sample, test_size=TEST_SIZE, random_state=SEED)
    neg_train, neg_test = train_test_split(neg_sample, test_size=TEST_SIZE, random_state=SEED)

    rows = (
        [(condition, u, 1, "train") for u in pos_train] +
        [(condition, u, 1, "test")  for u in pos_test]  +
        [(condition, u, 0, "train") for u in neg_train] +
        [(condition, u, 0, "test")  for u in neg_test]
    )
    return rows

manifest_rows = []
for condition in CONDITIONS:
    manifest_rows.extend(build_condition_split(condition))

manifest = pd.DataFrame(manifest_rows, columns=["condition", "user_id", "y", "split"])
manifest.shape

(9184, 4)

## 5. Sanity checks

In [9]:
# 1) no user appears in both train and test for the same condition
leak = (
    manifest.groupby(["condition", "user_id"])["split"].nunique()
)
assert (leak == 1).all(), "leakage: some user is in both train and test for a condition"

# 2) every condition's train/test is 1:1 balanced
balance = manifest.groupby(["condition", "split", "y"]).size().unstack("y").fillna(0)
balance.columns = ["n_control", "n_diagnosed"]
assert (balance["n_control"] == balance["n_diagnosed"]).all(), "train/test not balanced"

print("No leakage. All splits balanced 1:1.")
balance

No leakage. All splits balanced 1:1.


n_control  n_diagnosed
condition       split                        
adhd            test          65           65
                train        578          578
anxiety         test          65           65
                train        579          579
autism          test          23           23
                train        204          204
bipolar         test           8            8
                train         68           68
bpd             test          10           10
                train         88           88
depression      test         184          184
                train       1647         1647
eating_disorder test           5            5
                train         36           36
ocd             test          32           32
                train        279          279
panic           test          15           15
                train        132          132
ptsd            test          10           10
                train         82           82
schizophrenia   test           5            5
                train         40           40
sleep_disorder  test          36           36
                train        321          321
suicidal        test           8            8
                train         72           72

In [10]:
summary = (
    manifest.groupby("condition")
    .apply(lambda g: pd.Series({
        "n_users_total": g["user_id"].nunique(),
        "n_train": (g["split"] == "train").sum(),
        "n_test": (g["split"] == "test").sum(),
    }))
    .sort_values("n_users_total", ascending=False)
)
summary

,n_users_total,n_train,n_test
condition,,,
depression,3662,3294,368
anxiety,1288,1158,130
adhd,1286,1156,130
sleep_disorder,714,642,72
ocd,622,558,64
autism,454,408,46
panic,294,264,30
bpd,196,176,20
ptsd,184,164,20


## 6. Save the manifest

This is the artifact `experiments/classical.py` and `experiments/finetune.py` will be updated to consume (instead of each script independently re-deriving its own split), once approved.

In [11]:
manifest_path = OUT_DIR / "condition_user_splits.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Saved {len(manifest):,} (condition, user) rows to {manifest_path}")

Saved 9,184 (condition, user) rows to ../data/splits/condition_user_splits.csv
